<a href="https://colab.research.google.com/github/RhyanMezaroba/AI_Automation/blob/main/Aula_03_Reactive_Vacuum_Challenge_COLAB_ALUNO_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Aula 3 — Arquiteturas de Agentes
# 🧹 Reactive Vacuum Challenge

**Inteligência Artificial — UNOESC · Ciência da Computação**

**Pergunta central: como um agente transforma uma percepção em uma ação?**

Vamos começar com 2 salas, observar um agente de referência, descobrir sua política e depois construir nosso próprio agente. No final, ampliaremos para 5 salas no **Reactive Vacuum Challenge**.

# 🧭 Roteiro

1. Vacuum World clássico — 2 salas
2. Implementação guiada do agente reativo
3. Teste exaustivo dos 8 possíveis estados
4. Extensão para 5 salas (Reactive Vacuum Challenge)
5. Desenvolvimento de uma nova política
6. Cenários públicos
7. Avaliação secreta pela professora
8. Discussão das limitações da arquitetura

# 01 · 🏠 Vacuum World clássico — 2 salas

Ambiente:

```text
┌──────────┬──────────┐
│    A     │    B     │
└──────────┴──────────┘
```
Cada sala pode estar `Clean` ou `Dirty`.

Ações: `Suck`, `Left`, `Right`, `NoOp`.

A percepção do agente é:

```text
(posição atual, estado da sala atual)
```

Exemplos de percepções possíveis:

```text
(A, Dirty)
(A, Clean)
(B, Dirty)
(B, Clean)
```

A ideia central:

```text
PERCEPÇÃO → REGRA → AÇÃO
```

In [ ]:
MAX_STEPS = 30

In [ ]:
# =============================== #
# 🔒 INFRAESTRUTURA — 2 SALAS 🔒 #
# =============================== #

from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from itertools import product
import pandas as pd
import matplotlib.pyplot as plt

Position = str
Condition = str
Perception = Tuple[Position, Condition]

POSITIONS = ["A", "B"]
CONDITIONS = ["Clean", "Dirty"]
ACTIONS = ["Suck", "Left", "Right", "NoOp"]

@dataclass
class VacuumWorld:
    # Estado de cada sala. Ex.: {"A": "Dirty", "B": "Clean"}
    status: Dict[Position, Condition]
    # Posição atual do agente
    position: Position
    # Registro da execução pelo simulador
    actions: List[str] = field(default_factory=list)
    steps: int = 0

    def perceive(self) -> Perception:
        # O agente percebe somente sua posição e o estado da sala atual
        return self.position, self.status[self.position]

    def is_clean(self) -> bool:
        return all(value == "Clean" for value in self.status.values())

    def execute(self, action: str) -> None:
        self.actions.append(action)
        self.steps += 1

        if action == "Suck":
            self.status[self.position] = "Clean"
        elif action == "Left" and self.position == "B":
            self.position = "A"
        elif action == "Right" and self.position == "A":
            self.position = "B"


def run_episode(agent, start_position, status_map, max_steps=MAX_STEPS):
    world = VacuumWorld(dict(status_map), start_position)

    for _ in range(max_steps):
        # 1. Perceber
        perception = world.perceive()

        # 2. Decidir
        action = agent(perception)

        # 3. Agir
        world.execute(action)

    return world


def all_states():
    # 2 posições iniciais × 2² configurações de Clean/Dirty = 8 estados.
    for start in POSITIONS:
        for values in product(CONDITIONS, repeat=2):
            yield start, dict(zip(POSITIONS, values))


print("✅ Vacuum World de 2 salas carregado.")

# 02 · ✍️ Sua vez — agente de 2 salas

Implemente o agente.

### Regras

✅ usar apenas a percepção atual  
✅ aspirar quando estiver sujo  
✅ mover para a outra sala quando estiver limpo

❌ memória  
❌ busca  
❌ planejamento

Descubra a política do agente antes de continuar:

| Percepção | Ação *a* ser tomada | Por que essa ação? |
|---|---|---|
| `(A, Dirty)` | ? | ? |
| `(A, Clean)` | ? | ? |
| `(B, Dirty)` | ? | ? |
| `(B, Clean)` | ? | ? |

In [ ]:
# ===========================================================
# 🧑‍💻 ÁREA DO ALUNO
# ============================================================

def meu_agente(percepcao: Perception) -> str:
    # TODO 1: se estiver sujo, retorne "Suck".
    # TODO 2: se estiver limpo, escolha "Right" ou "Left".
    return "NoOp"

##Primeiro teste: Verifique se sua função realmente transforma percepção → ação

In [ ]:
test_perceptions = [
    ("A", "Dirty"), ("A", "Clean"),
    ("B", "Dirty"), ("B", "Clean")
]

for perception in test_perceptions:
    print(perception, "→", meu_agente(perception))

### Agora vamos executar um exemplo completo

In [ ]:
demo = run_episode(
    meu_agente,
    start_position="A",
    status_map={"A": "Dirty", "B": "Dirty"}
)

print("Ações:", demo.actions)
print("Estado final:", demo.status)
print("Sucesso:", demo.is_clean())
print("Passos:", demo.steps)

# 03 · 🧪 Teste exaustivo — 8 estados

Vamos verificar todos os estados iniciais possíveis.

Esse é nosso primeiro **checkpoint**: o agente precisa resolver o problema clássico antes de aumentar a dificuldade.

In [ ]:
rows = []

for episode, (start, status_map) in enumerate(all_states(), start=1):
    world = run_episode(meu_agente, start, status_map)

    rows.append({
        "episódio": episode,
        "início": start,
        "mapa": status_map,
        "sucesso": world.is_clean(),
        "passos": world.steps,
        "ações": world.actions,
    })

results = pd.DataFrame(rows)
display(results)

print(f"Resultado: {results['sucesso'].sum()} / {len(results)} estados resolvidos.")

## ✅ Checkpoint 1

> **O agente conseguiu limpar todos os cenários?**

> **Em quais precisou de mais ações?**

> **Existem ações desnecessárias?**

> **O seu agente pode ser otimizado?**

# 04 · 🌎 Agora complicamos o ambiente

Novo ambiente:

```text
┌──────────┬──────────┬──────────┬──────────┬──────────┐
│    A     │    B     │    C     │    D     │    E     │
└──────────┴──────────┴──────────┴──────────┴──────────┘
```

In [ ]:
# ============================================================
# 🔒 INFRAESTRUTURA — 5 SALAS
# ============================================================

Position5 = str
Perception5 = Tuple[Position5, Condition]

POSITIONS_5 = ["A", "B", "C", "D", "E"]

@dataclass
class VacuumWorld5:
    status: Dict[Position5, Condition]
    position: Position5
    actions: List[str] = field(default_factory=list)
    steps: int = 0
    invalid_actions: int = 0

    def perceive(self) -> Perception5:
        # única informação entregue ao agente: sua posição + estado da sala atual
        return self.position, self.status[self.position]

    def is_clean(self) -> bool:
        return all(value == "Clean" for value in self.status.values())

    def execute(self, action: str) -> None:
        self.actions.append(action)
        self.steps += 1

        if action not in ACTIONS:
            self.invalid_actions += 1
            return

        if action == "Suck":
            self.status[self.position] = "Clean"
            return

        index = POSITIONS_5.index(self.position)

        if action == "Left":
            if index == 0:
                self.invalid_actions += 1
            else:
                self.position = POSITIONS_5[index - 1]

        elif action == "Right":
            if index == len(POSITIONS_5) - 1:
                self.invalid_actions += 1
            else:
                self.position = POSITIONS_5[index + 1]


def run_episode_5(agent, start_position, status_map, max_steps=MAX_STEPS):
    world = VacuumWorld5(dict(status_map), start_position)

    for _ in range(max_steps):
        world.execute(agent(world.perceive()))

    return world


def describe_world_5(world):
    parts = []

    for position in POSITIONS_5:
        icon = "🦠" if world.status[position] == "Dirty" else "✨"
        robot = "🤖" if world.position == position else ""
        parts.append(f"{position} {icon}{robot}")

    print("  ↔  ".join(parts))
    print("Percepção:", world.perceive())
    print()

# 05 · ✍️ Desenvolva sua política para 5 salas

A percepção continua sendo:

```python
(position, status)
```

Você precisa decidir:

> **Quando a sala atual está limpa, para onde o agente deve ir?**

In [ ]:
# ============================================================
# 🧑‍💻 ÁREA DO ALUNO — 5 SALAS
# ============================================================

def meu_agente_5(percepcao: Perception5) -> str:
    # TODO 1: se estiver sujo, limpe.
    # TODO 2: quando estiver limpo, escolha uma política de movimento.
    return "NoOp"

# 08 · 🧪 Cenários públicos

Estes cenários são conhecidos.

Use-os para **debugging e desenvolvimento**.

⚠️ Atenção

Nem todo cenário pode ser resolvido perfeitamente por um agente reativo sem memória.

Em alguns casos, duas situações diferentes produzem exatamente a mesma percepção, mas exigem ações diferentes.

**Seu objetivo é investigar essa limitação e construir uma política eficiente — não necessariamente obter 100% de sucesso.**

In [ ]:
PUBLIC_SCENARIOS = [
    ("P01", "C", {"A":"Clean","B":"Dirty","C":"Clean","D":"Clean","E":"Clean"}),
    ("P02", "C", {"A":"Clean","B":"Clean","C":"Clean","D":"Dirty","E":"Clean"}),
    ("P03", "C", {"A":"Dirty","B":"Clean","C":"Clean","D":"Clean","E":"Clean"}),
    ("P04", "C", {"A":"Clean","B":"Clean","C":"Clean","D":"Clean","E":"Dirty"}),
    ("P05", "C", {"A":"Clean","B":"Dirty","C":"Clean","D":"Dirty","E":"Clean"}),
    ("P06", "B", {"A":"Dirty","B":"Clean","C":"Dirty","D":"Clean","E":"Dirty"}),
]

public_rows = []

for scenario_id, start, status_map in PUBLIC_SCENARIOS:
    world = run_episode_5(meu_agente_5, start, status_map)

    describe_world_5(world)

    public_rows.append({
        "Cenário": scenario_id,
        "Sucesso": world.is_clean(),
        "Passos": world.steps,
        "Ações inválidas": world.invalid_actions,
    })

public_results = pd.DataFrame(public_rows)

display(public_results)
print(f"Taxa de sucesso pública: {public_results['Sucesso'].mean():.1%}")
print(f"Média de passos: {public_results['Passos'].mean():.1f}")

# 09 · 🚀 Reactive Vacuum Challenge

## Agora começa a competição

Até aqui:

✅ 2 salas  
✅ agente clássico  
✅ 8 testes exaustivos  
✅ 5 salas  
✅ cenários públicos  
✅ debugging

### Agora os cenários mudam.

A professora possui uma bateria de **ambientes secretos**.

Você não verá os mapas usados na avaliação final.

> **Seu objetivo é criar uma política que generalize para ambientes que você nunca viu.**

## Regras

O agente final deve:

- receber somente `(posição, condição)`;
- não consultar o mapa completo;
- não guardar histórico;
- não usar busca;
- não planejar uma sequência futura;
- retornar ações válidas.

### Ranking

**1. Taxa de sucesso**  
**2. Menor média de passos nos casos resolvidos**  
**3. Menos ações inválidas**

> **Primeiro resolver. Depois otimizar.**

# 📦 Entrega

## 🏆 Para o ranking

Entregue diretamente à professora apenas a implementação do agente:

`meu_agente_5(percepcao)`

O agente será executado nos cenários de avaliação para composição do ranking.

```python
def meu_agente_5(percepcao):
    ...
```

## 📝 Registro no AVA

Registre:

1. Política utilizada
2. Justificativa da política
3. Resultados públicos
   - taxa de sucesso;
   - média de passos;
   - ações inválidas.
4. Dificuldades encontradas
5. Limitações do agente reativo
6. O que mudaria se o agente tivesse memória

Anexe o Colab completo utilizado na aula.